# Propagation Test

This notebook analyzes the propagation mechanisms of our optical knot sorter.


In [ ]:
# Functions to import

import numpy as np 
import scipy as sp
import pygad
import yaml 
from pathlib import Path

from scipy.fft import fft2, fftfreq, ifft2, fftshift, ifftshift
from scipy import ndimage
from optical_functions import TotInt, LG, propFF, propTF, cart2pol, oamModes, output_chan, setKnotType, output_chan_symmetric, output_chan_triangle, output_chan_circle, norm_field, build_fresnel_lens_kernels, propagate_fresnel_lens_train, propagate_legacy_fft, propagate_legacy_fft_supersampled, balanced_detector_throughput, complex_field_fidelity, intensity_fidelity
from sorter_configuration import parse_optical_train_config
#from run_ga import compute_sorting_performance

import matplotlib.pyplot as plt 

import os

# Physical Constants

nm = 1e-9
um = 1e-6
mm = 1e-3
cm = 1e-2

Load the optimized phase planes

In [ ]:
import pickle

index = 3

#experiment_name = "Knot Sorting Bread"
#experiment_name = "Knot Sorting New FF"
#experiment_name = "Knot Sorting Three Knot Fun"
#experiment_name = "Knot Sorting New FF Smaller Alpha"
#experiment_name = "Knot Sorting New FF Large Alpha"
#experiment_name = "tref_cinque_fresnel_1plane"

experiment_name = None  # configs/ga4.yaml: two-plane legacy FFT reference
 
config_path = Path('configs') / (f'ga{index}.yaml' if experiment_name is None else f'{experiment_name}/ga{index}.yaml')
with config_path.open('r', encoding='utf-8') as stream:
    cnfg = yaml.safe_load(stream)

cnfg.setdefault('circle_radius', 1.5)
cnfg.setdefault('alpha', 0.0)

N = cnfg['dim']
num_of_output_chans = cnfg['num_output_chans']
output_chan_width = cnfg['output_chan_width'] * mm # in mm 

num_phase_maps_near = cnfg.get('num_phase_maps_near', 0)
num_phase_maps_far = cnfg.get('num_phase_maps_far', 0)

num_of_phase_maps = cnfg.get('num_phase_planes', num_phase_maps_near + num_phase_maps_far)
optical_train = parse_optical_train_config(cnfg, num_of_phase_maps)
instance_name = cnfg['ga_instance'] # directory name of best phases

# Print the instance name (for reference)

print(instance_name)

# Some parameters specifying the LG modes

LG_modes = cnfg['LG_modes']
w0 = cnfg['w0'] * mm # in mm!!

isKnot = cnfg['isKnot']
knotType = cnfg['knotType']
shapeParams = cnfg['shapeParams']
fourier_lens = cnfg.get('fourier_length', 10.0)*cm # legacy propagation only
GFilterStrength=cnfg['gauss_filter_sigma']
channel_seperation = cnfg['channel_sep']
circle_radius = cnfg['circle_radius'] # circle radius is in mm
alpha = cnfg['alpha']

# Define the coordinate space 

la = cnfg.get('wavelength_nm', 780.0)*nm
k=(2*np.pi)/la  # [m^-1] wavenumber    
N = cnfg['dim'] # [Number of points per dimension]
maxx = cnfg.get('pixel_pitch_um', 20.0)*um*N  # Full numerical-window length (m)

# Propagation Distance 
prop_dist = 0

# Let's apply a rotation
rot_phi = eval(cnfg['rot_angle'])

# Space definition 
dx = maxx/N
dy = maxx/N 

#okay let's just say h here is dx or dy for now WLOG (WITH ... loss of generality)

h = dx
X = dx*(np.arange(N) - N //2)
Y = dy*(np.arange(N) - N //2)

# Apply rotation operator on coords 

xx,yy=np.meshgrid(X ,Y)

r, phi= cart2pol(xx, yy)

# What experiment are we going to design

simulateLens = cnfg.get('simulateLens', False)
multiPhase = cnfg.get('multiPhase', False)
multiPhaseLens = cnfg.get('multiPhaseLens', False)

z_o = cnfg.get('z_o', 30.0)*cm
fourier_lens = cnfg.get('fourier_length', 10.0)*cm

''' 
Create the OAM beams that we need to sort 
'''
# Now create a list containing 'oamMode' objects 

list_of_OAMs = []

output_chans = output_chan_circle(X, Y, output_chan_width, maxx, num_of_output_chans, circle_radius=circle_radius, coordinate_mode=optical_train.output_coordinate_mode)

if(isKnot):
    for ii in range(len(knotType)):
        field = setKnotType(r, phi, w0, knotType[ii], shapeParams[ii])
        prop_field = field
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
else:
    for ii in range(len(LG_modes)):
        ell, p = LG_modes[ii][0], LG_modes[ii][1]
        field = LG(r, phi, ell, p, w0, h, 0, k)
        prop_field = propTF(field, maxx, la, prop_dist)
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))


# Load up phase screens

with open(f"best_phases/{instance_name}.pkl", 'rb') as file:
     phase_out = pickle.load(file)

geometry_path = Path('best_phases') / f'{instance_name}_geometry.yaml'
analysis_padding_factor = optical_train.padding_factor
if optical_train.model == 'fresnel_lens_train' and geometry_path.exists():
    with geometry_path.open('r', encoding='utf-8') as stream:
        saved_geometry = yaml.safe_load(stream)
    analysis_padding_factor = saved_geometry.get('padding_factor', analysis_padding_factor)
    sorter_stages = [
        {'z_to_lens': stage['z_to_lens_cm']*cm,
         'focal_length': stage['focal_length_cm']*cm,
         'z_after_lens': stage['z_after_lens_cm']*cm}
        for stage in saved_geometry['stages']
    ]
else:
    initial_geometry = optical_train.initial_normalized_geometry if optical_train.num_geometry_genes else None
    sorter_stages = optical_train.decode_geometry(initial_geometry)
    if optical_train.model == 'fresnel_lens_train':
        print(f'Warning: {geometry_path} was not found; using the YAML initial geometry, not candidate-optimized geometry.')


phase_maps = np.empty((num_of_phase_maps, N, N), dtype=np.complex128)

# Compute phase screens

for ii in range(num_of_phase_maps):
    phase_maps[ii] = np.exp(1j * phase_out[ii])

analysis_fresnel_kernels = None
if optical_train.model == 'fresnel_lens_train':
    analysis_fresnel_kernels = build_fresnel_lens_kernels(
        (N, N), maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        padding_factor=analysis_padding_factor,
    )

def propagate_analysis_field(field):
    if optical_train.model != 'fresnel_lens_train':
        raise RuntimeError('propagate_analysis_field is for the physical Fresnel train.')
    return propagate_fresnel_lens_train(
        field, phase_maps, maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        kernels=analysis_fresnel_kernels,
        padding_factor=analysis_padding_factor,
    )


Radial Power Spectrum of Phase Masks

In [ ]:
for ii in range(num_of_phase_maps):
    S = np.abs(np.fft.fftshift(np.fft.fft2(phase_maps[ii])))**2
    n = S.shape[0]
    g = np.arange(n) - n//2
    RR = np.hypot(*np.meshgrid(g, g))
    half = S[RR < n/4].sum() / S.sum()
    print(f"{half:.1%} of the mask power is below half Nyquist")


## Legacy FFT sampling-convergence test

`samples_per_pixel` refines the computational sampling without changing the physical width of an input/image-plane device pixel. The incident knot or LG mode is rebuilt analytically on every refined grid. Input/image-plane masks are pixel-replicated; Fourier-plane masks retain their native `N x N` footprint and block light outside the device. Detector masks follow the same rule according to the parity of the train. The default `detector_mode='device'` preserves the published detector discretization; `detector_mode='physical'` instead rebuilds ideal circular apertures on the computed output pitch. The FFT directions follow the repository convention (forward, inverse, forward, ...), and `samples_per_pixel=1` is checked against the existing legacy propagation result.

In [ ]:
def rebuild_input_mode(input_index, axis, sample_pitch):
    if not 0 <= input_index < len(list_of_OAMs):
        raise IndexError('input_index is outside the configured alphabet.')
    grid_x, grid_y = np.meshgrid(axis, axis)
    grid_r, grid_phi = cart2pol(grid_x, grid_y)
    if isKnot:
        return setKnotType(
            grid_r, grid_phi, w0, knotType[input_index],
            shapeParams[input_index],
        )
    ell, radial_index = LG_modes[input_index]
    return LG(
        grid_r, grid_phi, ell, radial_index, w0, sample_pitch, 0, k
    )


def sampling_detector_channels(samples_per_pixel, detector_pitch, mode='device'):
    up = int(samples_per_pixel)
    grid_size = N*up
    if mode == 'device':
        if num_of_phase_maps % 2 == 0:
            return np.repeat(np.repeat(output_chans, up, axis=1), up, axis=2)
        channels = np.zeros(
            (num_of_output_chans, grid_size, grid_size), dtype=np.complex128
        )
        start = (grid_size-N)//2
        channels[:, start:start+N, start:start+N] = output_chans
        return channels
    if mode == 'physical':
        axis = detector_pitch*(np.arange(grid_size)-grid_size//2)
        return output_chan_circle(
            axis, axis, output_chan_width, grid_size*detector_pitch,
            num_of_output_chans, circle_radius=circle_radius,
            coordinate_mode='physical',
        )
    raise ValueError("mode must be 'device' or 'physical'.")


def published_legacy_efficiency():
    efficiency = np.zeros((len(list_of_OAMs), num_of_output_chans))
    for input_index, input_mode in enumerate(list_of_OAMs):
        field = np.asarray(input_mode.oamBeam, dtype=np.complex128)
        input_power = np.sum(np.abs(field)**2)
        output = propagate_legacy_fft(field, phase_maps)
        output_power = np.sum(np.abs(output)**2)
        output = output*np.sqrt(input_power/output_power)
        intensity = np.abs(output)**2
        for channel_index, channel in enumerate(output_chans):
            efficiency[input_index, channel_index] = (
                np.sum(intensity*np.real(channel))/input_power
            )
    return efficiency


def evaluate_sampling(samples_per_pixel, detector_mode='device'):
    if optical_train.model != 'legacy_fft':
        raise RuntimeError('This convergence test is for the legacy FFT train.')
    if multiPhase or multiPhaseLens or simulateLens:
        raise RuntimeError(
            'Use the direct legacy configuration: multiPhase, '
            'multiPhaseLens, and simulateLens must all be false.'
        )
    if int(samples_per_pixel) != samples_per_pixel or samples_per_pixel < 1:
        raise ValueError('samples_per_pixel must be a positive integer.')

    up = int(samples_per_pixel)
    grid_size = N*up
    input_pitch = h/up
    input_axis = input_pitch*(np.arange(grid_size)-grid_size//2)
    fourier_pitch = la*fourier_lens/(grid_size*input_pitch)
    detector_pitch = (
        fourier_pitch if num_of_phase_maps % 2 else input_pitch
    )
    channels = sampling_detector_channels(
        up, detector_pitch, mode=detector_mode
    )

    efficiency = np.zeros((len(list_of_OAMs), num_of_output_chans))
    survival = np.zeros(len(list_of_OAMs))
    stage_survival = []
    for input_index in range(len(list_of_OAMs)):
        field = rebuild_input_mode(input_index, input_axis, input_pitch)
        input_power = np.sum(np.abs(field)**2)
        output, records = propagate_legacy_fft_supersampled(
            field, phase_maps, samples_per_pixel=up,
            return_intermediate=True,
        )
        intensity = np.abs(output)**2
        survival[input_index] = intensity.sum()/input_power
        stage_survival.append([
            np.sum(np.abs(record['after_mask'])**2)/input_power
            for record in records
        ])
        for channel_index, channel in enumerate(channels):
            efficiency[input_index, channel_index] = (
                np.sum(intensity*np.real(channel))/input_power
            )

    accepted = efficiency.sum(axis=1, keepdims=True)
    assignment = np.divide(
        efficiency, accepted, out=np.zeros_like(efficiency),
        where=accepted > 0,
    )
    correct = np.diag(assignment)
    wrong_mean = (assignment.sum(axis=1)-correct)/(len(correct)-1)
    contrasts = correct-wrong_mean
    sorting_performance = alpha*np.min(contrasts)+(1-alpha)*np.mean(contrasts)
    throughput, accepted_efficiencies = balanced_detector_throughput(
        efficiency, method=cnfg.get('throughput_metric', 'geometric_mean')
    )
    return {
        'samples_per_pixel': up,
        'grid_size': grid_size,
        'input_pitch': input_pitch,
        'fourier_pitch': fourier_pitch,
        'detector_pitch': detector_pitch,
        'efficiency_matrix': efficiency,
        'assignment_matrix': assignment,
        'sorting_performance': float(sorting_performance),
        'throughput': throughput,
        'accepted_efficiencies': accepted_efficiencies,
        'survival': survival,
        'stage_survival': np.asarray(stage_survival),
    }


def run(up, input_index=0, detector_mode='device'):
    result = evaluate_sampling(up, detector_mode=detector_mode)
    return result['efficiency_matrix'][input_index]


In [ ]:
published_efficiency = published_legacy_efficiency()
sampling_results = [evaluate_sampling(up) for up in (1, 2, 4, 8, 16)]

unit_error = np.max(np.abs(
    sampling_results[0]['efficiency_matrix']-published_efficiency
))
np.testing.assert_allclose(
    sampling_results[0]['efficiency_matrix'], published_efficiency,
    rtol=1e-11, atol=1e-13,
)
print(f'up=1 regression maximum error: {unit_error:.3e}')

for result in sampling_results:
    print('\n' + '-'*60)
    print(
        f"up={result['samples_per_pixel']}, "
        f"grid={result['grid_size']} x {result['grid_size']}, "
        f"input pitch={result['input_pitch']/um:.3f} um, "
        f"detector pitch={result['detector_pitch']/um:.3f} um"
    )
    print('Efficiency matrix:')
    print(result['efficiency_matrix'])
    print('Assignment matrix:')
    print(result['assignment_matrix'])
    print(f"Sorting performance: {result['sorting_performance']:.8g}")
    print(f"Balanced throughput: {result['throughput']:.8g}")
    print('Total propagated-power survival:', result['survival'])

ups = np.asarray([result['samples_per_pixel'] for result in sampling_results])
sorting_curve = np.asarray([result['sorting_performance'] for result in sampling_results])
throughput_curve = np.asarray([result['throughput'] for result in sampling_results])
survival_curve = np.asarray([np.min(result['survival']) for result in sampling_results])

figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(ups, sorting_curve, 'o-', label='Sorting performance')
axes[0].set(xlabel='Samples per device pixel', ylabel='Conditional score')
axes[0].grid(alpha=0.25)
axes[1].plot(ups, throughput_curve, 's-', label='Detector throughput')
axes[1].plot(ups, survival_curve, '^-', label='Minimum power survival')
axes[1].set(xlabel='Samples per device pixel', ylabel='Power fraction', ylim=(0, 1.01))
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.show()
